### qwen-model

In [ ]:
## write the full model architecture here

In [ ]:
## rope
import torch

def compute_rope_angles(head_dim, theta_base=10000, context_length=2048, dtype=torch.float32):
    
    assert head_dim // 2 == 0 ## head_dim should be even

    index = torch.arange(0, head_dim, 2, dtype=dtype)
    inv_freq = 1.0 / (theta_base ** (2 * index / head_dim))

    ## compute positions
    positions = torch.arange(context_length, dtype=dtype) ## we are calculating the positions here  

    ## computing the angle 
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0) ### positions = 2048, 1  ** 1 ---- inv_freq = head_dim // 2 -----> angles = 2048 * 64

   
    cos = torch.cos(angles) ## 2048, 64
    sin = torch.sin(angles)  ## 2048, 64




In [ ]:
def apply_rope(x, cos, sin):

    B, T, H, D = x.shape

    x1 = x[..., ::2]   # [B, T, H, D/2]
    x2 = x[..., 1::2]  # [B, T, H, D/2]

    # Handle both training and inference
    if cos.dim() == 2:  # training
        cos = cos.unsqueeze(0).unsqueeze(2)  # [1, T, 1, D/2]
        sin = sin.unsqueeze(0).unsqueeze(2)
    else:               # inference (single position)
        cos = cos[None, None, None, :]       # [1, 1, 1, D/2]
        sin = sin[None, None, None, :]

    x_even = x1 * cos - x2 * sin
    x_odd  = x1 * sin + x2 * cos

    x_out = torch.stack([x_even, x_odd], dim=-1).flatten(-2) # (b, t, h, d/2, 2) --> # (b, t, h, d)

    return x_out  # (b, t, h, d)

In [ ]:
## RMSNorm
import torch
import torch.nn as nn

class RMSNorm(nn.module):
    def __init__(self, emb, eps=1e-6):
        super().__init__()

        self.eps = eps
        self.weight = nn.Parameter(torch.ones(emb))

    def forward(self, x):
        square = torch.square(x)
        sq_mean = square.mean(dim=-1, keepdim=True)  #b,t,1
        value = sq_mean +  self.eps
        rms_value = torch.sqrt(value)
        normalized_value = x / rms_value
        value = normalized_value * self.weight  ## b,t,d -- ## b, t, 1 
        return value   # b, t, d


    

In [ ]:
import torch
import torch.nn as nn

class GroupQueryAttention(nn.Module):
    def __init__(self, d_in, num_heads, kv_heads, head_dim, dtype=torch.float32):
        super().__init__()

        self.d_in = d_in
        self.num_heads = num_heads
        self.kv_heads = kv_heads
        self.group_size = num_heads // kv_heads  # scalar
        self.head_dim = head_dim

        self.d_out = num_heads * head_dim

        ## projections 
        self.w_query  = nn.Linear(self.d_in, self.d_out, dtype=dtype)
        self.w_keys   = nn.Linear(self.d_in, self.kv_heads * self.head_dim, dtype=dtype)
        self.w_values = nn.Linear(self.d_in, self.kv_heads * self.head_dim, dtype=dtype)
        

        self.out_proj = nn.Linear(self.d_out, self.d_in, dtype=dtype)

        ## norms
        self.q_norm = RMSNorm(self.d_out)                  # (b, t, num_heads * head_dim)
        self.k_norm = RMSNorm(kv_heads * head_dim)         # (b, t, kv_heads * head_dim)


    def forward(self, x, cos, sin, mask, start_pos=0, cache=None):
        b, t, _ = x.shape                                  # (b, t, d_in)

        ## projections 
        queries = self.w_query(x)                           # (b, t, num_heads * head_dim)
        keys    = self.w_keys(x)                            # (b, t, kv_heads * head_dim)
        values  = self.w_values(x)                          # (b, t, kv_heads * head_dim)

        ## reshape (RoPE layout)
        queries  = queries.view(b, t, self.num_heads, self.head_dim)     # (b, t, num_heads, d)
        keys_new = keys.view(b, t, self.kv_heads, self.head_dim)         # (b, t, kv_heads, d)
        values_new = values.view(b, t, self.kv_heads, self.head_dim)     # (b, t, kv_heads, d)

        ## RoPE (expects [b, t, h, d])
        queries  = apply_rope(queries,  cos, sin, offset=start_pos)      # (b, t, num_heads, d)
        keys_new = apply_rope(keys_new, cos, sin, offset=start_pos)      # (b, t, kv_heads, d)

        ## transpose to cache/attention layout
        queries    = queries.transpose(1, 2)             # (b, num_heads, t, d)
        keys_new   = keys_new.transpose(1, 2)            # (b, kv_heads, t, d)
        values_new = values_new.transpose(1, 2)          # (b, kv_heads, t, d)

        ## cache
        if cache is not None:
            prev_k, prev_v = cache                       # (b, kv_heads, t_prev, d)

            keys   = torch.cat([prev_k, keys_new], dim=2)    # (b, kv_heads, t_prev + t, d)
            values = torch.cat([prev_v, values_new], dim=2)  # (b, kv_heads, t_prev + t, d)
        else:
            keys, values = keys_new, values_new              # (b, kv_heads, t, d)

        next_cache = (keys, values)                      # tuple((b, kv_heads, T, d), (b, kv_heads, T, d))

        ## expand KV to match query heads
        keys   = keys.repeat_interleave(self.group_size, dim=1)    # (b, num_heads, T, d)
        values = values.repeat_interleave(self.group_size, dim=1)  # (b, num_heads, T, d)

        ## attention 

        attn_scores = queries @ keys.transpose(2, 3)              # (B, Hq, T, D) @ (B, Hq, D, T) → (B, Hq, T, T)

        attn_scores = attn_scores / (self.head_dim ** 0.5)        # scale logits (B, Hq, T, T)

        attn_scores = attn_scores.masked_fill(mask, -torch.inf)   # mask  (B, Hq, T, T)

        attn_weights = torch.softmax(attn_scores, dim=-1)         # softmax over T_k → (B, Hq, T, T)

        # Apply attention to values

        context = attn_weights @ values                           # (B, Hq, T, T) @ (B, Hq, T, D) → (B, Hq, T, D)

        
        # Merge heads

        context = context.transpose(1, 2)                         # (B, Hq, T, D) → (B, T, Hq, D)

        context = context.reshape(b, t, self.d_out)               # (B, T, Hq*D)

        # Output projection

        out = self.out_proj(context)                              # (B, T, D_model)
                

